In [1]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_aapl
!git status

/content
Cloning into 'ddpm_option_pricing'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 195 (delta 36), reused 56 (delta 18), pack-reused 121 (from 1)
Receiving objects: 100% (195/195), 87.38 MiB | 19.61 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/ddpm_option_pricing
Fetching origin
Branch 'v2_aapl' set up to track remote branch 'v2_aapl' from 'origin'.
Switched to a new branch 'v2_aapl'
On branch v2_aapl
Your branch is up to date with 'origin/v2_aapl'.

nothing to commit, working tree clean


In [2]:
import numpy as np
from src_real.data.prices_yf import load_prices_yfinance
from src_real.data.blocks import (
    log_returns_from_prices,
    build_return_blocks,
    standardize_blocks_per_dim,
)

prices = load_prices_yfinance("AAPL", start="2023-06-01", end="2024-06-01", price_col="Adj Close")

rets = log_returns_from_prices(prices)
H = 21
dt = 1/252

rb = build_return_blocks(returns=rets, H=H, dt=dt, stride=1)
Z = standardize_blocks_per_dim(rb)

print("Z mean/std:", float(Z.mean()), float(Z.std(ddof=1)))
print("per-dim std first 5:", Z.std(axis=0, ddof=1)[:5])

Z mean/std: -4.718240464995915e-09 0.9979360103607178
per-dim std first 5: [0.99999994 1.         1.         0.99999994 0.9999999 ]


In [3]:
import torch
from src_real.diffusion.schedules import make_alpha_schedule
from src_real.models.score_mlp_vec import ScoreMLPVec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

T_diff = 1000
betas, alphas, alphas_bar = make_alpha_schedule(
    T=T_diff,
    device=device,
    schedule="linear",
    beta_max=2e-2,
)

print("alphas_bar[-1]:", float(alphas_bar[-1]))
print("betas min/max:", float(betas.min()), float(betas.max()))

model = ScoreMLPVec(H=H, hidden_dim=256, time_emb_dim=64).to(device)

alphas_bar[-1]: 4.035827805637382e-05
betas min/max: 9.999999747378752e-05 0.019999999552965164


In [4]:
from torch.utils.data import DataLoader, TensorDataset
from src_real.diffusion.train import train_ddpm_blocks

x_train = torch.from_numpy(Z).float()
loader = DataLoader(TensorDataset(x_train), batch_size=64, shuffle=True, drop_last=True)

stats = train_ddpm_blocks(
    model=model,
    train_loader=loader,
    alphas_bar=alphas_bar,
    T=T_diff,
    device=device,
    epochs=80,
    lr=1e-3,
)

print("first 5 losses:", stats.losses[:5])
print("last 5 losses :", stats.losses[-5:])

Epoch 1/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 2/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 3/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 4/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 5/80:   0%|          | 0/3 [00:00<?, ?it/s]

...

Epoch 77/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 78/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 79/80:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 80/80:   0%|          | 0/3 [00:00<?, ?it/s]

first 5 losses: [1.0116621255874634, 0.9537445306777954, 0.9275261163711548, 0.9967010617256165, 0.9070383906364441]
last 5 losses : [0.2833609879016876, 0.2605143189430237, 0.23637822270393372, 0.29134294390678406, 0.37136074900627136]


In [6]:
from src_real.diffusion.sample import sample_blocks_ddpm

Z_samp = sample_blocks_ddpm(
    model=model,
    n_samples=5000,
    H=21,
    alphas=alphas,
    alphas_bar=alphas_bar,
    betas=betas,
    device=device,
    chunk=5000,
    temperature=1.1,
)

print("Z_train mean/std:", float(Z.mean()), float(Z.std(ddof=1)))
print("Z_samp  mean/std:", float(Z_samp.mean()), float(Z_samp.std(ddof=1)))
print("Z_samp per-dim std first 5:", Z_samp.std(axis=0, ddof=1)[:5])

Z_train mean/std: -4.718240464995915e-09 0.9979360103607178
Z_samp  mean/std: -0.009638848714530468 1.0030995607376099
Z_samp per-dim std first 5: [1.0077546  1.0127363  1.028202   0.9724801  0.94866353]


In [7]:
import numpy as np
from src_real.data.blocks import unstandardize_blocks_per_dim

X_train = rb.blocks.astype(np.float64)  # real return blocks (N,H)
X_samp  = unstandardize_blocks_per_dim(Z_samp, rb).astype(np.float64)

print("TRAIN mean/std:", float(X_train.mean()), float(X_train.std(ddof=1)))
print("SAMP  mean/std:", float(X_samp.mean()),  float(X_samp.std(ddof=1)))

print("TRAIN per-dim std first 5:", X_train.std(axis=0, ddof=1)[:5])
print("SAMP  per-dim std first 5:",  X_samp.std(axis=0, ddof=1)[:5])

print("TRAIN min/max:", float(X_train.min()), float(X_train.max()))
print("SAMP  min/max:", float(X_samp.min()),  float(X_samp.max()))

TRAIN mean/std: 8.640112830816005e-05 0.012963810307982243
SAMP  mean/std: -3.59628210913191e-05 0.013030948004202384
TRAIN per-dim std first 5: [0.01243844 0.01301189 0.01301626 0.01301779 0.01300788]
SAMP  per-dim std first 5: [0.0125349  0.01317762 0.01338336 0.01265954 0.0123401 ]
TRAIN min/max: -0.04921133443713188 0.058095671236515045
SAMP  min/max: -0.05404702574014664 0.05302916467189789


In [8]:
import numpy as np, math
from src_real.finance.pricing_smile import (
    terminal_prices_from_return_blocks,
    price_calls_from_ST,
    smile_table_from_prices,
)

S0 = float(prices[-1])
r = 0.04
dt = 1/252
H = 21
T_total = H * dt

Ks_over = np.array([0.8, 0.9, 1.0, 1.1, 1.2], dtype=float)
Ks = Ks_over * S0

ST_P = terminal_prices_from_return_blocks(X_samp, S0=S0)
pP, seP = price_calls_from_ST(ST_P, Ks=Ks, r=r, T=T_total)

dfP = smile_table_from_prices(S0=S0, Ks=Ks, prices=pP, stderrs=seP, r=r, T=T_total)
dfP["label"] = "DDPM_P"

print(dfP)

            K  K_over_S0      price    stderr        iv   label
0  152.762451        0.8  38.264167  0.162100       NaN  DDPM_P
1  171.857758        0.9  19.404800  0.157173       NaN  DDPM_P
2  190.953064        1.0   4.681071  0.098000  0.198437  DDPM_P
3  210.048370        1.1   0.291841  0.022928  0.202917  DDPM_P
4  229.143677        1.2   0.004191  0.002711  0.204444  DDPM_P


In [11]:
import numpy as np, math
from src_real.finance.pricing_smile import bs_smile_table

sigma_anchor = float(np.std(rets, ddof=1) / math.sqrt(dt))  # annualized from daily log rets
dfBS = bs_smile_table(S0=S0, Ks=Ks, r=r, T=T_total, sigma=sigma_anchor)
dfBS["label"] = "BS_flat"

print("sigma_anchor:", sigma_anchor)
print(dfBS)

sigma_anchor: 0.20231366226799122
            K  K_over_S0      price  stderr        iv    label
0  152.762451        0.8  38.699095     0.0  0.202314  BS_flat
1  171.857758        0.9  19.796264     0.0  0.202314  BS_flat
2  190.953064        1.0   4.766012     0.0  0.202314  BS_flat
3  210.048370        1.1   0.287805     0.0  0.202314  BS_flat
4  229.143677        1.2   0.003700     0.0  0.202314  BS_flat


In [12]:
import numpy as np
import math
from src_real.finance.pricing_smile import (
    price_calls_from_ST,
    smile_table_from_prices,
)

# ---- parameters ----
n_mc = X_samp.shape[0]   # same MC size as DDPM
sigma = sigma_anchor    # from earlier
mu = r                  # risk-neutral drift

# ---- simulate GBM terminal prices ----
Z = np.random.randn(n_mc)
ST_gbm = S0 * np.exp(
    (mu - 0.5 * sigma**2) * T_total
    + sigma * math.sqrt(T_total) * Z
)

# ---- price options ----
pG, seG = price_calls_from_ST(
    ST_gbm,
    Ks=Ks,
    r=r,
    T=T_total,
)

dfGBM = smile_table_from_prices(
    S0=S0,
    Ks=Ks,
    prices=pG,
    stderrs=seG,
    r=r,
    T=T_total,
)
dfGBM["label"] = "GBM_MC"

print(dfGBM)

            K  K_over_S0      price    stderr        iv   label
0  152.762451        0.8  38.816563  0.158976  0.373025  GBM_MC
1  171.857758        0.9  19.916577  0.155073  0.228870  GBM_MC
2  190.953064        1.0   4.871274  0.099012  0.207118  GBM_MC
3  210.048370        1.1   0.285981  0.023256  0.202039  GBM_MC
4  229.143677        1.2   0.004810  0.003192  0.206866  GBM_MC


In [13]:
import numpy as np
import math

# assumes you already have:
# X_samp  : (n_mc, H) sampled log-return blocks in data space
# rb      : ReturnBlocks from build_return_blocks(...)
# r, dt, sigma_anchor

# sigma_anchor (annualized) if you don't have it:
sigma_anchor = float(np.std(rets, ddof=1) / math.sqrt(dt))
print("sigma_anchor:", sigma_anchor)

# risk-neutral daily mean for log-returns (GBM drift)
mQ = (r - 0.5 * sigma_anchor**2) * dt   # scalar

# empirical per-day mean vector from training blocks
mP_vec = rb.mean_vec.astype(np.float64)  # (H,)

# mean-shifted "Q" blocks
X_Q_mean = X_samp.astype(np.float64) + (mQ - mP_vec[None, :])

print("mQ:", mQ)
print("mP_vec first 5:", mP_vec[:5])
print("X_Q_mean mean/std:", float(X_Q_mean.mean()), float(X_Q_mean.std(ddof=1)))

sigma_anchor: 0.20231366226799122
mQ: 7.751821837244679e-05
mP_vec first 5: [-1.56063688e-04  7.48103703e-05  6.81060337e-05  9.34416166e-05
  1.35211871e-04]
X_Q_mean mean/std: -4.4845755462836265e-05 0.013030295105401052


In [14]:
import numpy as np

def exp_martingale_projection(X_blocks, r, dt):
    """
    Shift each day j by delta_j so that mean(exp(X_j + delta_j)) = exp(r dt).
    This enforces one-step RN condition marginally for each day.
    """
    X = np.asarray(X_blocks, dtype=np.float64)
    target = np.exp(r * dt)
    deltas = np.log(target) - np.log(np.mean(np.exp(X), axis=0))  # (H,)
    X_proj = X + deltas[None, :]
    return X_proj, deltas

X_proj, deltas = exp_martingale_projection(X_samp, r=r, dt=dt)

print("deltas first 5:", deltas[:5])
print("X_proj mean/std:", float(X_proj.mean()), float(X_proj.std(ddof=1)))

deltas first 5: [ 0.00197289 -0.00060474  0.00122788 -0.00030722 -0.00070289]
X_proj mean/std: 7.456963112996312e-05 0.012974618894152069


In [15]:
import numpy as np
from src_real.finance.pricing_smile import (
    terminal_prices_from_return_blocks,
    price_calls_from_ST,
    smile_table_from_prices,
)

# strikes already defined like you did:
# Ks_over, Ks, S0, T_total

# --- Q mean shift pricing ---
ST_Q_mean = terminal_prices_from_return_blocks(X_Q_mean, S0=S0)
pQm, seQm = price_calls_from_ST(ST_Q_mean, Ks=Ks, r=r, T=T_total)

dfQ_mean = smile_table_from_prices(S0=S0, Ks=Ks, prices=pQm, stderrs=seQm, r=r, T=T_total)
dfQ_mean["label"] = "DDPM_Q_mean_shift"
print(dfQ_mean)

print()

# --- exp projection pricing ---
ST_proj = terminal_prices_from_return_blocks(X_proj, S0=S0)
pPr, sePr = price_calls_from_ST(ST_proj, Ks=Ks, r=r, T=T_total)

dfProj = smile_table_from_prices(S0=S0, Ks=Ks, prices=pPr, stderrs=sePr, r=r, T=T_total)
dfProj["label"] = "DDPM_exp_proj"
print(dfProj)

            K  K_over_S0      price    stderr        iv              label
0  152.762451        0.8  38.228631  0.162069       NaN  DDPM_Q_mean_shift
1  171.857758        0.9  19.370728  0.157107       NaN  DDPM_Q_mean_shift
2  190.953064        1.0   4.662547  0.097812  0.197591  DDPM_Q_mean_shift
3  210.048370        1.1   0.289623  0.022830  0.202586  DDPM_Q_mean_shift
4  229.143677        1.2   0.004157  0.002697  0.204300  DDPM_Q_mean_shift

            K  K_over_S0      price    stderr        iv          label
0  152.762451        0.8  38.706908  0.162476  0.272923  DDPM_exp_proj
1  171.857758        0.9  19.830534  0.157972  0.210971  DDPM_exp_proj
2  190.953064        1.0   4.915506  0.100344  0.209137  DDPM_exp_proj
3  210.048370        1.1   0.320371  0.024173  0.207066  DDPM_exp_proj
4  229.143677        1.2   0.004626  0.002884  0.206171  DDPM_exp_proj
